In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
df=pd.read_csv(r"C:\Users\GR0012AU\Downloads\train.csv")

In [3]:
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
665,666,0,2,"Hickman, Mr. Lewis",male,32.0,2,0,S.O.C. 14879,73.5000,NaN,S
155,156,0,1,"Williams, Mr. Charles Duane",male,51.0,0,1,PC 17597,61.3792,NaN,C
204,205,1,3,"Cohen, Mr. Gurshon ""Gus""",male,18.0,0,0,A/5 3540,8.0500,NaN,S
317,318,0,2,"Moraweck, Dr. Ernest",male,54.0,0,0,29011,14.0000,NaN,S
376,377,1,3,"Landergren, Miss. Aurora Adelia",female,22.0,0,0,C 7077,7.2500,NaN,S


In [4]:
df1=df.drop(columns=["PassengerId","Name","Ticket","Cabin"])

In [5]:
df1["family"]=df1["SibSp"]+df1["Parch"]

In [6]:
df1=df1.drop(columns=["SibSp","Parch"])

In [7]:
df1

,Survived,Pclass,Sex,Age,Fare,Embarked,family
0,0,3,male,22.0,7.2500,S,1
1,1,1,female,38.0,71.2833,C,1
2,1,3,female,26.0,7.9250,S,0
3,1,1,female,35.0,53.1000,S,1
4,0,3,male,35.0,8.0500,S,0
...,...,...,...,...,...,...,...
886,0,2,male,27.0,13.0000,S,0
887,1,1,female,19.0,30.0000,S,0
888,0,3,female,NaN,23.4500,S,3
889,1,1,male,26.0,30.0000,C,0


In [8]:
dependent_column=df1["Survived"]
independent_column=df1.drop(columns=["Survived"])

In [9]:
dependent_column

0      0
1      1
2      1
3      1
4      0
      ..
886    0
887    1
888    0
889    1
890    0
Name: Survived, Length: 891, dtype: int64

In [10]:
independent_column

,Pclass,Sex,Age,Fare,Embarked,family
0,3,male,22.0,7.2500,S,1
1,1,female,38.0,71.2833,C,1
2,3,female,26.0,7.9250,S,0
3,1,female,35.0,53.1000,S,1
4,3,male,35.0,8.0500,S,0
...,...,...,...,...,...,...
886,2,male,27.0,13.0000,S,0
887,1,female,19.0,30.0000,S,0
888,3,female,NaN,23.4500,S,3
889,1,male,26.0,30.0000,C,0


In [11]:
from sklearn.model_selection import train_test_split

In [12]:
x_train,x_test,y_train,y_test=train_test_split(independent_column,dependent_column,test_size=0.3,random_state=0)

In [13]:
y_test

495    0
648    0
278    0
31     1
255    1
      ..
263    0
718    0
620    0
786    1
64     0
Name: Survived, Length: 268, dtype: int64

In [14]:
from  sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.feature_selection import SelectKBest,chi2
from sklearn.tree import DecisionTreeClassifier

In [15]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Sex       891 non-null    object 
 3   Age       714 non-null    float64
 4   Fare      891 non-null    float64
 5   Embarked  889 non-null    object 
 6   family    891 non-null    int64  
dtypes: float64(2), int64(3), object(2)
memory usage: 48.9+ KB


In [16]:
x_train.isnull().sum()

Pclass        0
Sex           0
Age         121
Fare          0
Embarked      2
family        0
dtype: int64

In [17]:
trf1=ColumnTransformer([
    ("Impute_Age",SimpleImputer(),[2]),
    ("Impute_Embarked",SimpleImputer(strategy="most_frequent"),[4])
],remainder="passthrough")

In [18]:
trf2=ColumnTransformer([
    ("OHE_sex_embarked",OneHotEncoder(drop="first",sparse=False,handle_unknown="ignore"),[3,1])
],remainder="passthrough")

In [19]:
trf3=ColumnTransformer([
    ("scaler",MinMaxScaler(),slice(0,7))
],remainder="passthrough")

In [20]:
trf4=SelectKBest(score_func=chi2,k=6)

In [21]:
trf5=DecisionTreeClassifier()

In [22]:
pipe=Pipeline([
    ("trf1",trf1),
    ("trf2",trf2),
    ("trf3",trf3),
    ("trf4",trf4),
    ("trf5",trf5)
])

In [23]:
pipe.fit(x_train,y_train)

Pipeline(steps=[('trf1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('Impute_Age', SimpleImputer(),
                                                  [2]),
                                                 ('Impute_Embarked',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [4])])),
                ('trf2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('OHE_sex_embarked',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse=False),
                                                  [3, 1])])),
                ('trf3',
                 ColumnTransformer(remainder='passthrough',
           

In [24]:
# pipe.named_steps
# pipe.named_steps["trf1"]
# pipe.named_steps["trf1"].transformers_
# pipe.named_steps["trf1"].transformers_[0]
# pipe.named_steps["trf1"].transformers_[0][1]
pipe.named_steps["trf1"].transformers_[0][1].statistics_

array([29.91533865])

In [25]:
y_pred=pipe.predict(x_test)

In [26]:
from sklearn.metrics import accuracy_score

In [27]:
acc_sc=accuracy_score(y_test,y_pred)

In [28]:
print(acc_sc)

0.7835820895522388


In [29]:
from sklearn.model_selection import cross_val_score
cross_val_score(pipe,x_train,y_train,cv=5,scoring="accuracy").mean()

0.7655870967741937

In [31]:
import pickle
pickle.dump(pipe,open("Titanic_model_pipeline.pkl","wb"))